# v2 Persona Vector Pipeline — Extraction (Colab / Kaggle)

Runs the GPU-heavy extraction pass once per model: sub-test A + sub-test B prompt activations (all layers, both pooling variants, both formatting variants where available), plus Method 1's generation activations for instruct models. Resume-safe throughout — if this session gets evicted, just re-run from the top; already-completed work is never recomputed (see `src/checkpoint.py`).

**Prerequisites before this notebook can run:**
1. `Pipeline_v2/` and the `data/` CSVs (including `tone_pole_*.csv`, `subtest_b_*.csv`) must be pushed to the repo's remote — this notebook clones the repo fresh. Update `REPO_URL` below if the remote differs from `github.com/Fjord-H/Persona-Vector-Study`.
2. An `HF_TOKEN` secret (Colab: *Secrets* panel; Kaggle: *Add-ons → Secrets*) with license acceptance on file for `meta-llama/Llama-3.2-3B` and `meta-llama/Llama-3.2-3B-Instruct` — both are gated.
3. Nothing else — the unit tests (including the required unbatched-equivalence test) run automatically below and must pass before extraction starts.

In [1]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

REPO_URL = "https://github.com/Fjord-H/Persona-Vector-Study.git"  # update if the remote differs
REPO_DIR = pathlib.Path("/content/Persona-Vector-Study") if IN_COLAB else pathlib.Path("/kaggle/working/Persona-Vector-Study")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

PIPELINE_DIR = REPO_DIR / "Pipeline_v2"
sys.path.insert(0, str(PIPELINE_DIR))
os.chdir(PIPELINE_DIR)

# requirements.txt deliberately does NOT list torch -- Kaggle/Colab GPU images ship a
# pre-installed, CUDA-matched PyTorch build, and a plain pip install of a pinned torch
# version can silently pull a mismatched or CPU-only wheel. Only the pure-Python /
# no-CUDA-binary packages are pinned and (re)installed here.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("environment:", "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "unknown/local"))
print("working dir:", pathlib.Path.cwd())

import torch, transformers, huggingface_hub, numpy, pandas, sklearn
print()
print("installed versions:")
for name, mod in [("torch", torch), ("transformers", transformers), ("huggingface_hub", huggingface_hub),
                   ("numpy", numpy), ("pandas", pandas), ("scikit-learn", sklearn)]:
    print(f"  {name:16s} {mod.__version__}")

print()
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print(
        "WARNING: no CUDA GPU visible to torch yet. Fine for the setup/unit-test cells "
        "below (they run on CPU), but check the notebook's Accelerator setting "
        "(Kaggle: Settings -> Accelerator -> GPU; Colab: Runtime -> Change runtime type) "
        "before running the extraction cell -- CPU extraction works but is far slower "
        "for Qwen/Llama."
    )

Already up to date.
environment: Kaggle
working dir: /kaggle/working/Persona-Vector-Study/Pipeline_v2

installed versions:
  torch            2.10.0+cu128
  transformers     5.0.0
  huggingface_hub  1.11.0
  numpy            2.0.2
  pandas           2.3.3
  scikit-learn     1.6.1

torch.cuda.is_available(): True


In [2]:
# Cache directory: Colab -> Google Drive (survives eviction), Kaggle -> /kaggle/working
# (persists for the session and can be saved as Kaggle output), local -> Pipeline_v2/cache.
# Overridable at any time by setting PV2_CACHE_DIR yourself before importing src.config.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    cache_root = pathlib.Path("/content/drive/MyDrive/persona_vector_v2_cache")
elif IN_KAGGLE:
    cache_root = pathlib.Path("/kaggle/working/pv2_cache")
else:
    cache_root = PIPELINE_DIR / "cache"

cache_root.mkdir(parents=True, exist_ok=True)
os.environ["PV2_CACHE_DIR"] = str(cache_root)
os.environ["PV2_MANIFEST_DIR"] = str(cache_root.parent / "pv2_manifests")
print("cache dir:", cache_root)
# Point data dir at the Kaggle dataset mount
if IN_KAGGLE:
    os.environ["PV2_DATA_DIR"] = "/kaggle/input/datasets/fjordhauler/pipeline-v2-data"
    os.environ["PV2_SPLITS_JSON"] = "/kaggle/input/datasets/fjordhauler/pipeline-v2-data/splits.json"

print("data dir:", os.environ.get("PV2_DATA_DIR", "not set"))

cache dir: /kaggle/working/pv2_cache
data dir: /kaggle/input/datasets/fjordhauler/pipeline-v2-data


In [3]:
# HF token — required for the gated Llama-3.2 models (base + instruct).
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print("Could not read HF_TOKEN from Colab secrets:", e)
elif IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Could not read HF_TOKEN from Kaggle secrets:", e)

assert os.environ.get("HF_TOKEN"), (
    "HF_TOKEN not set. Add it as a Colab/Kaggle secret named HF_TOKEN -- required for "
    "meta-llama/Llama-3.2-3B and -Instruct, both gated."
)
print("HF_TOKEN is set (", len(os.environ["HF_TOKEN"]), "chars )")

HF_TOKEN is set ( 37 chars )


## Required: unit tests must pass before any real extraction

Per the spec's compute-constraints section — the unbatched-equivalence test in particular (`tests/test_pooling_batch_equivalence.py`) is the single highest-probability-of-a-new-bug spot given the length-channel finding in this project's data. Do not skip this cell.

In [4]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], cwd=PIPELINE_DIR)
assert result.returncode == 0, "Tests failed -- DO NOT proceed to real extraction until every test passes."

....................................                                     [100%]
=============================== warnings summary ===============================
<frozen importlib._bootstrap>:488
  <frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute

<frozen importlib._bootstrap>:488
  <frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
36 passed, 2 warnings in 254.26s (0:04:14)


sys:1: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


In [5]:
# Frozen split: computed once (seeded, stratified, near-duplicate-grouped) and reused
# from here on. Safe to re-run -- it's a no-op once data/splits.json exists.
from src import splits
from src.config import SPLITS_JSON_PATH

splits_df = splits.load_frozen_splits()
print("frozen splits ready at", SPLITS_JSON_PATH)
print(splits_df.groupby(["label", "split"]).size().unstack(fill_value=0))

frozen splits ready at /kaggle/input/datasets/fjordhauler/pipeline-v2-data/splits.json
split    test  train  val
label                    
harmful    35    175   40
neutral    41    217   42


## Extraction

Edit `MODELS_TO_RUN` to extract a subset (useful for a first smoke run — start with `["gpt2-medium"]`, the smallest and only non-gated model, before spending quota on Qwen/Llama). Each model is fully resume-safe: re-running this cell after an eviction picks up exactly where it left off, per model and per formatting variant, with zero recomputation of already-cached items.

In [6]:
import torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Use both T4s if available
print("GPUs available:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i), 
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f}GB")

GPUs available: 2
  GPU 0: Tesla T4 15.6GB
  GPU 1: Tesla T4 15.6GB


In [7]:
from src.config import MODEL_REGISTRY
from src.run_extraction import extract_all_for_model

MODELS_TO_RUN = list(MODEL_REGISTRY.keys())
# MODELS_TO_RUN = ["gpt2-medium"]  # uncomment for a first smoke run

def progress(total, already_done, remaining, **_):
    print(f"    {already_done}/{total} done, {remaining} remaining", end="\r")

for model_key in MODELS_TO_RUN:
    print(f"=== {model_key} ===")
    manifest_paths = extract_all_for_model(model_key, progress_callback=progress)
    print()
    for p in manifest_paths:
        print("  wrote manifest:", p)

=== gpt2-medium ===


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    616/616 done, 0 remaining
  wrote manifest: /kaggle/working/pv2_manifests/gpt2-medium__raw__2026-09-03T17-56-23.157356+00-00.json
=== qwen2.5-1.5b ===


...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

    616/616 done, 0 remaining
  wrote manifest: /kaggle/working/pv2_manifests/llama-3.2-3b__raw__2026-09-03T17-57-36.308644+00-00.json
=== llama-3.2-3b-instruct ===


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


    80/80 done, 0 remainingg
  wrote manifest: /kaggle/working/pv2_manifests/llama-3.2-3b-instruct__raw__2026-09-03T17-58-08.948793+00-00.json
  wrote manifest: /kaggle/working/pv2_manifests/llama-3.2-3b-instruct__chat__2026-09-03T17-58-22.324273+00-00.json
  wrote manifest: /kaggle/working/pv2_manifests/llama-3.2-3b-instruct__generation__2026-09-03T18-03-05.785262+00-00.json


## Done

Everything after this point (method comparison, bootstrap CIs) is pure numpy/sklearn on the cache written above — no further GPU time needed, and it can run anywhere (including locally, off this cache directory). See `notebooks/02_method_comparison.ipynb`.

If using Colab with a Drive-mounted cache, the cache already persists across sessions. If using Kaggle's `/kaggle/working`, remember to save this notebook's output/version so the cache directory is retained as Kaggle output before the session ends.